# Helico Structure Viewer

Load and visualize predicted PDB/CIF files from `helico-infer`.

In [1]:
import py3Dmol
from pathlib import Path

In [2]:
#%pip install py3Dmol

Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- Set this to your structure file ---
STRUCTURE_PATH = "../workbench/pred.pdb"

path = Path(STRUCTURE_PATH)
assert path.exists(), f"File not found: {path.resolve()}"
data = path.read_text()
fmt = "cif" if path.suffix in (".cif", ".mmcif") else "pdb"
print(f"Loaded {path.name} ({len(data)} bytes, format={fmt})")

Loaded pred.pdb (9156 bytes, format=pdb)


In [3]:
def show_structure(data: str, fmt: str = "pdb", style: str = "cartoon",
                   color: str = "plddt", width: int = 800, height: int = 600):
    """Render a structure with py3Dmol.

    Args:
        data: PDB or CIF file contents as string.
        fmt: "pdb" or "cif".
        style: "cartoon", "stick", "sphere", or "line".
        color: "plddt" (B-factor AlphaFold coloring), "chain", "spectrum",
               "element", or a hex color like "#88ccff".
        width, height: viewer dimensions in pixels.
    """
    view = py3Dmol.view(width=width, height=height)
    view.addModel(data, fmt)

    # Build color spec
    if color == "plddt":
        # AlphaFold pLDDT coloring via B-factor:
        #   >90 blue, 70-90 cyan, 50-70 yellow, <50 orange
        color_spec = {"prop": "b", "gradient": "roygb", "min": 0, "max": 100}
    elif color == "chain":
        color_spec = {"prop": "chain"}
    elif color == "spectrum":
        color_spec = "spectrum"
    elif color == "element":
        color_spec = {}
    else:
        color_spec = {"color": color}

    style_spec = {style: {"colorscheme": color_spec} if isinstance(color_spec, dict) and color_spec else {}}
    if color == "spectrum":
        style_spec = {style: {"color": "spectrum"}}
    elif color == "element":
        style_spec = {style: {}}
    elif isinstance(color_spec, dict):
        style_spec = {style: {"colorscheme": color_spec}}
    else:
        style_spec = {style: {"color": color}}

    view.setStyle(style_spec)
    view.zoomTo()
    return view.show()

## Cartoon — colored by pLDDT

In [4]:
show_structure(data, fmt, style="cartoon", color="plddt")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Stick — colored by element

In [5]:
show_structure(data, fmt, style="stick", color="element")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Cartoon + sticks (combined view)

In [6]:
view = py3Dmol.view(width=800, height=600)
view.addModel(data, fmt)
view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0, "max": 100}}})
view.addStyle({"stick": {"radius": 0.15}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Compare two structures side by side

In [ ]:
# Set to a second structure path (or None to skip)
COMPARE_PATH = None  # e.g. "../workbench/pred2.pdb"

if COMPARE_PATH is not None:
    path2 = Path(COMPARE_PATH)
    assert path2.exists(), f"File not found: {path2.resolve()}"
    data2 = path2.read_text()
    fmt2 = "cif" if path2.suffix in (".cif", ".mmcif") else "pdb"

    view = py3Dmol.view(width=800, height=600, linked=False, viewergrid=(1, 2))
    view.addModel(data, fmt, viewer=(0, 0))
    view.addModel(data2, fmt2, viewer=(0, 1))
    view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0, "max": 100}}},
                  viewer=(0, 0))
    view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0, "max": 100}}},
                  viewer=(0, 1))
    view.zoomTo(viewer=(0, 0))
    view.zoomTo(viewer=(0, 1))
    view.show()
else:
    print("Set COMPARE_PATH above to compare two structures.")